In [2]:
import pandas as pd
import numpy as np
import re
import time
import random
from openai import OpenAI
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [4]:
# Load questions dataset
questions_df = pd.read_excel("mathcamps_grade_2_sampled_100.xlsx")
skills_list = [
"Addition", "Subtraction", "Multiplication", "Division",
"Fractions",
"Solving Equations", "Word Problem Solving", "Geometry",
"Comparing Quantities", "Measurement", "Estimation"
]


# Initialize OpenAI
client = OpenAI(api_key="your_api_key_here")

FileNotFoundError: [Errno 2] No such file or directory: 'mathcamps_grade_2_sampled_100.xlsx'

In [ ]:
# Generate student knowledge profile
def generate_students(n_students):
    students = []
    temperatures = [0.0, 0.33, 0.66, 1.0]
    for i in range(n_students):
        skill_vector = np.random.choice([0, 1], size=len(skills_list))
        temp = temperatures[i % 4]
        students.append({
        'student_id': i+1,
        'known_skills': [skills_list[i] for i, val in enumerate(skill_vector) if val == 1],
        'unknown_skills': [skills_list[i] for i, val in enumerate(skill_vector) if val == 0],
        'skill_vector': skill_vector.tolist(),
        'temperature': temp
        })
    return students

In [ ]:
# Format student profile text
def build_student_profile(student):
    return f"""
    STUDENT KNOWLEDGE PROFILE:
    MASTERED SKILLS: {', '.join(student['known_skills'])}
    UNKNOWN / WEAK SKILLS: {', '.join(student['unknown_skills'])}
    """

In [ ]:
# Detect if question contains unknown skills
def detect_unknown_skills_vector(student, questions_df):
    vector = []
    for skills in questions_df['detected_skills']:
        q_skills = [s.strip() for s in skills.split(',')]
        vector.append(int(any(s in student['unknown_skills'] for s in q_skills)))
    return vector

In [ ]:
# Predict answer with given model and temp

def predict_student_answer(row, profile_text, model_name="gpt-4o", temperature=0.7):
    prompt = f"""
    You are a teacher predicting a student's answer to a math multiple-choice question, based on the student's knowledge profile.
    {profile_text}
    Here is the question:
    {row['Question']}
    A) {row['A']}
    B) {row['B']}
    C) {row['C']}
    D) {row['D']}
    Only return the letter you predict (A, B, C, or D).
    """
    try:
        time.sleep(1)
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=10,
            temperature=temperature,
            top_p=0.9
        )
        raw = response.choices[0].message.content.strip()
        return re.search(r"[ABCD]", raw.upper()).group()
    except:
        return "?"

In [ ]:
# Evaluate predictions

def evaluate_predictions(gt, pred):
    return [1 if g == p else 0 for g, p in zip(gt, pred)]

In [ ]:
# Build all experiments
models = ["gpt-4o", "gpt-4", "gpt-3.5-turbo"]
students = generate_students(120)
answers_vector = questions_df['Answers'].tolist()


student_model_objects = []


for student in students:
    profile_text = build_student_profile(student)
    unknown_vector = detect_unknown_skills_vector(student, questions_df)
    for model in models:
        preds = [predict_student_answer(row, profile_text, model_name=model, temperature=student['temperature'])
                for _, row in questions_df.iterrows()]
        evals = evaluate_predictions(answers_vector, preds)
        student_model_objects.append({
            'student_id': student['student_id'],
            'model': model,
            'temperature': student['temperature'],
            'known_skills': student['known_skills'],
            'unknown_skills': student['unknown_skills'],
            'answers_vector': preds,
            'questions_with_unknown_skills_vector': unknown_vector,
            'evaluation_vector': evals
        })

In [ ]:
# Export results
records = []
for entry in student_model_objects:
    for i, (pred, correct, unknown) in enumerate(zip(entry['answers_vector'], entry['evaluation_vector'], entry['questions_with_unknown_skills_vector'])):
        records.append({
            'student_id': entry['student_id'],
            'model': entry['model'],
            'temperature': entry['temperature'],
            'question_index': i+1,
            'predicted_answer': pred,
            'correct': correct,
            'expected_wrong': unknown
        })


experiment_df = pd.DataFrame(records)
experiment_df.to_excel("full_student_model_experiment_results.xlsx", index=False)